In [ ]:
from lsst.daf.butler import Butler
import pandas as pd
import numpy as np

import importlib

import smp_task_io
import sceneModel

In [ ]:
instrument = "LSSTCam"
repo = "dp2_prep"
collection_stage4 = "LSSTCam/runs/DRP/DP2"
butler_stage4 = Butler(repo, collections=collection_stage4)

## **Task I/O**

In [ ]:
importlib.reload(smp_task_io)

In [ ]:
#file_path = "/home/g/garazigp/rubin-user/scene_modeler/python/lsst/data_test/lightcurve_tract_2704_g_739161397740437575.csv"
file_path = "data_test/lightcurve_tract_2704_g_739161397740437575.csv"

df = pd.read_csv(file_path)
visits_to_load = set(zip(df["visitId"].astype(int), df["detectorId"].astype(int)))

In [ ]:
visit_image_refs = []
for visit, det in list(visits_to_load)[:20]:
    try:
        visitImage = butler_stage4.getDeferred("visit_image", visit=visit, detector=det, instrument="LSSTCam")
        #visit_images[(visit, det)] = visitImage
        visit_image_refs.append(visitImage)
        print(f"OK  visit={visit}  det={det}")
    except Exception as e:
        print(f"ERR visit={visit}  det={det}  → {e}")

print(f"\n{len(visit_image_refs)} images loaded.")
print(type(visit_image_refs[0]))

In [ ]:
for visit, det in list(visits_to_load)[:3]:
    ref = butler_stage4.registry.findDataset(
        'visit_image', visit=visit, detector=det, instrument='LSSTCam'
    )
    print(f"visit={visit} det={det} → run={ref.run}")

In [ ]:
tract = 2704
diaObjectId = 739161397740437575
diaObject = butler_stage4.get("dia_object", tract=tract, skymap="lsst_cells_v2")
print(type(diaObject))

In [ ]:
#diaObject.columns

In [ ]:
ra, dec = [diaObject.loc[diaObjectId]['ra']], [diaObject.loc[diaObjectId]['dec']]

In [ ]:
config = smp_task_io.SMPTaskConfig(ra=ra, dec=dec)
task = smp_task_io.SMPTask(config=config)

In [ ]:
data = task.run(visit_image_refs)

## **Test PSF derivative**

In [ ]:
# - _piff not existing => solution :
#         cp /opt/lsst/software/stack/conda/envs/lsst-scipipe-14.0.0-exact/lib/python3.13/site-packages/piff/_piff*.so ~/repos/Piff/piff/
#
# - version.py missing in meas_extensions_piff => solution :
#         cd ~/repos/meas_extensions_piff
#         setup -k -r . 
#         scons
#
# - To ensure that the version 1.7 (with PSF derivative from pfleget branch) of Piff is executed => solution : in .user_setups, before scene_modeler
#         export PYTHONPATH=$HOME/repos/Piff:$PYTHONPATH

In [ ]:
#import sys
#sys.path.insert(0, "/home/g/garazigp/repos/Piff")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import piff
import lsst.geom as geom
import lsst.meas.extensions.piff.piffPsf as pp

In [ ]:
# Check the loaded versions
print(piff.__file__, piff.version)
print(pp.__file__)
print("Piff a draw_derivative :", hasattr(piff.PSF, "draw_derivative"))
print("PiffPsf a computeImageDerivative :", hasattr(pp.PiffPsf, "computeImageDerivative"))

In [ ]:
visit = 2025091800199
visitImage = butler_stage4.get('visit_image', visit=visit, detector=113)

In [ ]:
# Check PSF basic properties
psf = visitImage.getPsf()
print("psf variable type :", type(psf))
model = psf.piffResult.model
print("Piff PSF model :", model)               # PixelGrid ? If not, will raise a NotImplementedError in the derivative
print("interpolant :", model.interp)           # Lanczos ?
print("interpolation properties :", psf.piffResult.interp_property_names)  # x ? y ? colorValue ?

In [ ]:
# Check PSF derivative computation
pos = visitImage.getBBox().getCenter()
img = psf.computeImage(pos)
dIdx, dIdy = psf.computeImageDerivative(pos)
print(pos)
print(visitImage.getBBox())
print(img.getBBox() == dIdx.getBBox(), img.getBBox() == dIdy.getBBox())
print(img.array.sum(), dIdx.array.sum(), dIdy.array.sum())

In [ ]:
# Compare derivative with 'différences finies' : the unitary test accepts 1e-4
h = 1e-3
pr = psf.piffResult

def draw(cx, cy):
    return pr.draw(pos.x, pos.y, stamp_size=psf.width, center=(cx, cy)).array

norm = draw(pos.x, pos.y).sum()
fdx = (draw(pos.x + h, pos.y) - draw(pos.x - h, pos.y)) / (2*h*norm)
fdy = (draw(pos.x, pos.y + h) - draw(pos.x, pos.y - h)) / (2*h*norm)

print("écart max dIdx / amplitude :", np.abs(dIdx.array - fdx).max() / np.abs(fdx).max())
print("écart max dIdy / amplitude :", np.abs(dIdy.array - fdy).max() / np.abs(fdy).max())

In [ ]:
# Visualise
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, arr, t in zip(ax, [img.array, dIdx.array, dIdy.array], ["PSF", "dI/dx", "dI/dy"]):
    im = a.imshow(arr, origin="lower"); a.set_title(t); plt.colorbar(im, ax=a)
plt.show()

In [ ]:
vmax = max(np.abs(dIdx.array).max(), np.abs(dIdy.array).max())

fig, ax = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
im0 = ax[0].imshow(img.array, origin="lower"); ax[0].set_title("PSF")
fig.colorbar(im0, ax=ax[0])

for a, arr, t in zip(ax[1:], [dIdx.array, dIdy.array], ["dI/dx", "dI/dy"]):
    im = a.imshow(arr, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    a.set_title(t)
fig.colorbar(im, ax=ax[1:], label="derivative amplitude")
plt.savefig('psf_and_derivative_example.png')
plt.show()

In [ ]:
pr = psf.piffResult

def check_position(pos, h=1e-3):
    img = psf.computeImage(pos)
    dIdx, dIdy = psf.computeImageDerivative(pos)

    def draw(cx, cy):
        return pr.draw(pos.x, pos.y, stamp_size=psf.width, center=(cx, cy)).array

    norm = draw(pos.x, pos.y).sum()
    fdx = (draw(pos.x + h, pos.y) - draw(pos.x - h, pos.y)) / (2*h*norm)     # compare to finite differences
    fdy = (draw(pos.x, pos.y + h) - draw(pos.x, pos.y - h)) / (2*h*norm)

    # centroïde de l'image en coordonnées du détecteur (vérifie le placement du stamp)
    b = img.getBBox()
    yy, xx = np.mgrid[b.getMinY():b.getMaxY() + 1, b.getMinX():b.getMaxX() + 1]
    cx = (xx * img.array).sum() / img.array.sum()
    cy = (yy * img.array).sum() / img.array.sum()

    return dict(
        x=pos.x, y=pos.y,
        bbox_ok=(b == dIdx.getBBox()) and (b == dIdy.getBBox()),
        err_x=np.abs(dIdx.array - fdx).max() / np.abs(fdx).max(),
        err_y=np.abs(dIdy.array - fdy).max() / np.abs(fdy).max(),
        sum_x=dIdx.array.sum(), sum_y=dIdy.array.sum(),
        amp_x=np.abs(fdx).max(), amp_y=np.abs(fdy).max(),
        dcx=cx - pos.x, dcy=cy - pos.y,
    )

def show(results):
    print(f"{'x':>8} {'y':>8} {'bbox':>5} {'err_x':>9} {'err_y':>9} {'sum_x':>9} {'sum_y':>9} {'dcx':>7} {'dcy':>7}")
    for r in results:
        print(f"{r['x']:8.1f} {r['y']:8.1f} {str(r['bbox_ok']):>5} {r['err_x']:9.2e} {r['err_y']:9.2e} "
              f"{r['sum_x']:9.1e} {r['sum_y']:9.1e} {r['dcx']:7.3f} {r['dcy']:7.3f}")

In [ ]:
bb = visitImage.getBBox()
xmax, ymax = bb.getMaxX(), bb.getMaxY()
rng = np.random.default_rng(42)

interior = [geom.Point2D(float(rng.uniform(100, xmax - 100)), float(rng.uniform(100, ymax - 100)))
            for _ in range(10)]
edges = [geom.Point2D(x, y) for x, y in [
    (0, 0), (xmax, 0), (0, ymax), (xmax, ymax),                       # coins
    (xmax/2, 0), (xmax/2, ymax), (0, ymax/2), (xmax, ymax/2),         # milieux des bords
]]
special = [geom.Point2D(1000.0, 1000.0), geom.Point2D(1000.5, 1000.5)]   # entier et demi-entier

print("--- inside ---");  show([check_position(p) for p in interior])
print("--- edges ---");      show([check_position(p) for p in edges])
print("--- integer or half-integer ---"); show([check_position(p) for p in special])

- Bigger error for integer coordinate
- 1 pixel shift for half-integer positions (cf. dcx, cdy)

In [ ]:
for x in [1000.5 - 1e-6, 1000.5, 1000.5 + 1e-6]:
    p = geom.Point2D(x, 1000.37)
    r = check_position(p)
    print(f"x = {x}   dcx = {r['dcx']:6.3f}   bbox minX = {psf.computeImage(p).getBBox().getMinX()}")

In [ ]:
hs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]
test_pos = [visitImage.getBBox().getCenter(), geom.Point2D(1000.5, 1000.5), geom.Point2D(0, 0)]
for p in test_pos:
    print(f"position ({p.x}, {p.y})")
    for h in hs:
        r = check_position(p, h=h)
        print(f"   h={h:7.0e}   err_x={r['err_x']:9.2e}   err_y={r['err_y']:9.2e}")

## **Scene Model**

In [ ]:
importlib.reload(sceneModel)

In [ ]:
data_arrays = []
weights = []
masks = []
psfs = []
wcss = []

for image_cutout in data.sceneModelingInputs['cutouts'] :
    data_arrays.append(image_cutout.data)
    weights.append(image_cutout.uncertainty.array**(-1))
    masks.append(image_cutout.flags)
    psfs.append(image_cutout.psf)
    wcss.append(image_cutout.wcs)

In [ ]:
#photo_ratio = np.ones(len(data_arrays))
position = data.sceneModelingInputs['positions'][0]
photo_ratio = data.sceneModelingInputs['photometric_ratios']
on_off = data.sceneModelingInputs['on_off']

model,resids = sceneModel.SceneModel(position, data_arrays, weights, masks, psfs,
                      wcss, photo_ratio, np.array(on_off), imageSize=51, modelSize=55)

In [ ]:
#init_params = np.zeros(55**2 + len(data_arrays) + np.array(data.sceneModelingInputs['on_off']).sum() + 2)
init_galaxy = np.zeros((55,55))
init_galaxy[25:28, 15:18] = 1
init_params = np.concatenate([init_galaxy.ravel(), np.zeros(len(data_arrays) + np.array(data.sceneModelingInputs['on_off']).sum() + 2)])
models, resids = model.generateAbChi2(init_params)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(3, 3, figsize=(8,8))
ax1[0].imshow(data_arrays[0])
ax1[1].imshow(models[0])
ax1[2].imshow(resids[0])

ax2[0].imshow(data_arrays[1])
ax2[1].imshow(models[1])
ax2[2].imshow(resids[1])

ax3[0].imshow(data_arrays[2])
ax3[1].imshow(models[2])
ax3[2].imshow(resids[2])

In [ ]:
wcs = wcss[0]
wcs.pixel_to_world(0,0)

## **Tests**

In [ ]:
import lsst.geom as geom

visitImage_test = butler_stage4.get('preliminary_visit_summary', visit=2025072300532, detector=154)

sky_coord = geom.SpherePoint(ra[0], dec[0], geom.degrees)
visitImage_wcs = visitImage_test.find(57).wcs  # detector 86 is outside
pixel_point = visitImage_wcs.skyToPixel(sky_coord)

print(pixel_point)

In [ ]:
len(model.xImage)

In [ ]:
REPO = "dp2_prep"
COLLECTION_STAGE2  = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2"
butler_stage2 = Butler(REPO, collections=COLLECTION_STAGE2)

In [ ]:
visits_test = df["visitId"].astype(int)[:20]
det_test = df["detectorId"].astype(int)[:20]
print(len(visits_test))

vsummaries = []

for v_id in visits_test:
    v_summary  = butler_stage2.get('visit_summary', visit=v_id).asAstropy()
    detector = v_summary[v_summary['visit'] == v_id]
    vsummaries.append(v_summary)

In [ ]:
test_tab = np.array([1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6])
print(int(len(test_tab)/4))
reshaped_tab = test_tab.reshape((int(len(test_tab)/4),4))
print(reshaped_tab)